# <u>Random Forest Regression</u>

### Prerequisites:
* <a href="../../1.Regression/7.Descision Trees Regression/Descision Trees Regression.ipynb">Check out the notebook on Decision Tree Regression</a>


## Topics

* [1. Core idea](#idea)
* [2. Bagging (Foundation of RF)](#bagging)
    * [**2.1 Regression**](#reg)
    * [2.2 Classification](#class)
        * <a href="../../2.Classification/7.Random Forest Classifier/Random Forest Classification.ipynb">Check out the notebook on Random Forest Classification</a>
* [3. What makes Random Forests different](#differ)
* [4. Performance Characteristics](#character)
* [5. Key Hyperparameters](#hyper)
* [6. Overfitting](#overfitting)
* [7. Out-of-Bag (OOB) Error](#oob)
* [8. Feature Importance](#important)
* [9. Proximities (Advanced Feature)](#proxi)
* [10. Advantages & Disadvantages](#adv_disadv)
* [11. Random Forest Regression library](#library)

    

In [3]:
import numpy as np # for rnadom number, linear algebra and general mathematical operations
import pandas as pd # for creating and working with dataframe
from matplotlib import pyplot as plt # for plotting in 2d
import plotly.express as px # for plotting in 3d
import plotly.graph_objects as go # for plotting in 3d
from sklearn.tree import DecisionTreeRegressor # for Regression trees
from sklearn.tree import plot_tree # for potting trees
from sklearn.tree import export_text # show tree rules
#from sklearn.preprocessing import OneHotEncoder # for one hot encoding of categoricals
#from sklearn.preprocessing import OrdinalEncoder # for ordinal labeling of categoricals with an order
from sklearn.impute import SimpleImputer # for replacing missing values
from sklearn.metrics import mean_squared_error # mean squarred error library
from sklearn.datasets import make_regression # to create toy data for regression
from sklearn.ensemble import RandomForestRegressor # for Random Forest Regressor
print("Setup complete")

Setup complete


<a class="anchor" id="idea"></a>
# 1. Core idea

- A Random Forest is an ensemble method that combines many decision trees
- It improves performance by:
    - Bagging (**Bootstrap Aggregation**) $\rightarrow$ training trees on different sampled datasets
    - Randomization $\rightarrow$ using random subsets of features at each split
- Works for both classification and regression

<p align="center">
<img src="pics/1.png" width="600"/>
</p>

---

<p align="center">
<img src="pics/2.png" width="600"/>
</p>

In [4]:
np.random.seed(1002) # reproducibility of random numbers
n = 100
X = np.random.uniform(0,10,(n,1))
y = np.sin(X.flatten()) + np.random.randn(n)*0.3


tree_model=DecisionTreeRegressor(max_depth=4,random_state=1008)
tree_model.fit(X,y)

fig = px.scatter(x=X.flatten(),y=y,title="Regression Tree Variability (poor performance)")
x_grid=np.linspace(np.min(X)-2,np.max(X)+2,100).reshape(-1,1)
pred_grid = tree_model.predict(x_grid)
fig.add_scatter(x=x_grid.flatten(), y=pred_grid, mode="lines", name="Original tree predictions")

for i in range(3):
    idx = np.random.choice(n,n,replace=True) # sample data with replacement
    X_changed = X[idx] # rearange data
    y_changed = y[idx] # reanrange data
    tree_model=DecisionTreeRegressor(max_depth=4,random_state=1008)
    tree_model.fit(X_changed,y_changed)
    pred_grid = tree_model.predict(x_grid)
    fig.add_scatter(x=x_grid.flatten(), y=pred_grid, mode="lines", name=f"Change data {i} time")


fig.show()

<a class="anchor" id="bagging"></a>
# 2. Bagging (Foundation of RF)

- Bagging means **B**ootstrap **Agg**regation
- Train multiple tree models (called Base learner) on bootstrap samples of training data $\mathcal{D}$ (sampling with replacement)
    - Draw $n$ observations from $\mathcal{D}$ with replacement
    - Fit Base learners/Tree models on each bootstrapped data $\mathcal{D}^{[m]}$ to obtain $m$ base learners $b^{[m]}$ 

<p align="center">
<img src="pics/7.png" width="600"/>
</p>

- Each model sees slightly different data:
    - In-bag data (IB): used for training 
    - Out-of-bag data (OOB): left out, used for validation


<p align="center">
<img src="pics/3.png" width="600"/>
</p>


#### Why Bagging it helps:
- Reduces variability of predictions by averaging the
outcomes from multiple Base learner models/ Tree models
- Works best for high-variance models like decision trees (in particular Classification trees)
- Increasing the ensemble size (number of Base learners) improves predictions stability even more (most for CLassification trees)


<p align="center">
<img src="pics/10.png" width="600"/>
</p>



<a class="anchor" id="reg"></a>
## **2.1 Regression**

<p align="center">
<img src="pics/8.png" width="600"/>
</p>

---

<p align="center">
<img src="pics/4.png" width="600"/>
</p>


In [ ]:
np.random.seed(1838)
n = 100
x = np.random.uniform(0,10,n)
y = np.sin(x) + np.random.rand(n)*0.9


fig = px.scatter(x=x,y=y,title="Average Predictions of multiple Trees")

tree_model=DecisionTreeRegressor(max_depth=4,random_state=1910)
tree_model.fit(x.reshape(n,1),y)
x_grid = np.linspace

for i in range(0,2):



fig.show()

<a class="anchor" id="class"></a>
## 2.2 Classification

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/14.png" width="350"/>
  <img src="pics/15.png" width="350"/>
  <img src="pics/16.png" width="350"/>
</div>


* <a href="../../2.Classification/7.Random Forest Classifier/Random Forest Classification.ipynb">Check out the notebook on Random Forest Classification for code</a>

<a class="anchor" id="differ"></a>
# 3. What makes Random Forests different

### Random Forest = Bagging + extra randomness

### Key addition on top of Bootstrap sampling:
- At each split, only a random subset of features is considered

<p align="center">
<img src="pics/6.png" width="600"/>
</p>

### Goal:
- Decorrelate trees with random feature sampling 
    - Trees that make mistakes in different directions
    - Example:

<p align="center">
<img src="pics/9_.jpeg" width="600"/>
</p>

- Reduce overall variance of the ensemble
- Fully expanded trees further increase variability of trees

<p align="center">
<img src="pics/12.png" width="600"/>
</p>

### Intuition for Decorrelated trees
- Since bootstrap samples are similar the base learners $\hat{b}^{[m]}$ are correlated which affects results in trees making similar mistakes
- Decorrelate the trees by randomizing the features considered for splits
    - For each node of tree, randomly draw a subset of the $p$ features
    - Only consider these features for finding the best split

<p align="center">
<img src="pics/13.png" width="600"/>
</p>

In [ ]:
np.random.seed(1838)
n = 100
X,y = make_regression(n_samples=n,n_features=1,bias=3,coef=False)
help(make_regression)

<a class="anchor" id="character"></a>
# 4. Performance Characteristics

- Increasing number of trees:
    - &#9989; stabilizes predictions
    - &#9989; reduces variance

- RF performs especially well for:
    - Classification tasks
    - High-dimensional data
    - Data with irrelevant/noisy features

<a class="anchor" id="hyper"></a>
# 5. Key Hyperparameters

- Number of trees (ntrees)
    - Typical: 100–500
- Number of features per split
    - Classification: $\lfloor \sqrt{p} \rfloor$
    - Regression: $\lfloor \frac{p}{3} \rfloor$
- Tree size
    - Usually fully grown trees (no pruning)
    - Controlled via:
        - min node size
        - max depth

```python
from sklearn.ensemble import RandomForestRegressor

rf1 = RandomForestRegressor(
    n_estimators=100, # number of trees
    criterion="squared_error", # "squared_error", "absolute_error"
    max_depth=None, # maximum depth of each tree
    min_samples_split=2, # min samples to split a node      
    min_samples_leaf=1, # min samples in a leaf
    max_features="sqrt", # number of features per split ("sqrt", "log2", or int)
    bootstrap=True, # use bootstrap sampling
    n_jobs=-1, # use all cores
    random_state=1801 # reproduce results
)

```

<a class="anchor" id="overfitting"></a>
# 6. Overfitting

- RFs are less prone to overfitting than single trees
- But still possible if:
    - Trees learn mostly noise
- Increasing tree depth:
    - $\downarrow$ variance
    - usually does **not** increase overfitting

<a class="anchor" id="oob"></a>
# 7. Out-of-Bag (OOB) Error


<p align="center">
<img src="pics/17.png" width="600"/>
</p>

---

<p align="center">
<img src="pics/18.png" width="600"/>
</p>

Tree 1,3 and tree 4 were not trained on observation $(x^{(2)},y^{(2)})=((\text{brown},\text{oblong},10,\text{Imported}),\text{yes})$ with ID 2 so it is for example 
$\text{OOB}^{[1]}=\{(x^{(2)},y^{(2)})\}=\{((\text{brown},\text{oblong},10,\text{Imported}),\text{yes}) \notin \mathcal{D}^{[1]}\}$.

---

<p align="center">
<img src="pics/11.png" width="600"/>
</p>

- Uses unused data per tree for validation
- Provides:
    - Built-in estimate of generalization error (error for future unseen data denoted as $\widehat{GE}$)
    - No need for separate validation set (untouched test data to pick hyperparameters)

- Used for:
    - Model evaluation
    - Hyperparameter tuning
    - Choosing number of trees

<a class="anchor" id="important"></a>
# 8. Feature Importance

### Helps interpret RFs (which are otherwise hard to interpret)

**Methods:**
- Permutation Importance

    - Shuffle a feature $\rightarrow$ measure drop in performance

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/21.png" width="550"/>
  <img src="pics/22_.png" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/23.png" width="550"/>
  <img src="pics/24.png" width="550"/>
</div>

- Impurity Importance

    - Sum improvement in splits using that feature

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/20.png" width="550"/>
  <img src="pics/19.png" width="550"/>
</div>

&#10071; Both can be biased toward features with many levels (i.e.,
continuous or categoricals with many categories)

<a class="anchor" id="proxi"></a>
# 9. Proximities (Advanced Feature)

- RF can measure similarity between data points
- Based on how often two points land in the same leaf

Applications:
- Visualization (clustering)
- Outlier detection (Outliers have low proximity to most observations)
- Missing value imputation (replace missing values with values from observations with close proximity)

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/25.png" width="550"/>
  <img src="pics/26.png" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/27.png" width="550"/>
  <img src="pics/28.png" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/29.png" width="550"/>
  <img src="pics/30.png" width="550"/>
</div>

<a class="anchor" id="adv_disadv"></a>
# 10. Advantages & Disadvantages

### &#9989; Advantages
- Strong predictive performance
- Handles:
    - high-dimensional data
    - missing values
    - noisy features
- Little preprocessing needed
- Parallelizable
- Overall same advantages as for Trees

### &#128680; Disadvantages
- Less interpretable than single trees
- Computationally expensive (many trees)
- Memory intensive
- Poor at extrapolation (like trees)

<p align="center">
<img src="pics/5.png" width="600"/>
</p>


### &#128273; Key Takeaway

Random Forests improve decision trees by:

**Averaging many decorrelated trees $\rightarrow$ lower variance + better generalization**


<a class="anchor" id="library"></a>
# 11. Random Forest Regression library

```python
# 1. Random Forest Regressor
from sklearn.ensemble import RandomForestRegressor

rf1 = RandomForestRegressor(
    n_estimators=100, # number of trees
    criterion="squared_error", # "squared_error", "absolute_error"
    max_depth=None, # maximum depth of each tree
    min_samples_split=2, # min samples to split a node      
    min_samples_leaf=1, # min samples in a leaf
    max_features="sqrt", # number of features per split ("sqrt", "log2", or int)
    bootstrap=True, # use bootstrap sampling
    n_jobs=-1, # use all cores
    random_state=1801 # reproduce results
)

rf1.fit(X, y)

rf1.predict(X) # predictions
rf1.score(X, y) # R^2 score



# 2. Random Forest with Cross-Validation (Grid Search)
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

rf2 = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    scoring="r2",
    cv=5,
    n_jobs=-1
)

rf2.fit(X, y)

rf2.best_params_ # best hyperparameters
rf2.best_score_ # best CV score
rf2.best_estimator_



# 3. Feature Importance
rf1.feature_importances_  # importance of each feature



# 4. Individual Tree Visualization
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

# plot one tree from the forest
plt.figure(figsize=(12, 8))
plot_tree(rf1.estimators_[0], filled=True)
plt.show()



# 5. Out-of-Bag (OOB) Score
rf_oob = RandomForestRegressor(
    n_estimators=100,
    oob_score=True,
    bootstrap=True,
    random_state=42
)

rf_oob.fit(X, y)

rf_oob.oob_score_ # OOB R^2 estimate



# 6. Randomized Search (faster tuning)
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "n_estimators": [50, 100, 200, 300],
    "max_depth": [None, 5, 10, 20, 30],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2"]
}

rf6 = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    scoring="r2",
    cv=5,
    random_state=42,
    n_jobs=-1
)

rf6.fit(X, y)

rf6.best_params_
rf6.best_score_


# 7. Train/Test Split Evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf7 = RandomForestRegressor(n_estimators=100, random_state=42)
rf7.fit(X_train, y_train)

y_pred = rf7.predict(X_test)

mean_squared_error(y_test, y_pred)
rf7.score(X_test, y_test)   # R^2



# 8. Pipeline (optional preprocessing)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline([
    ("scaler", StandardScaler()),  # not necessary for trees, but possible
    ("rf", RandomForestRegressor(n_estimators=100))
])

pipeline.fit(X, y)
pipeline.predict(X)



# 9. Compare with Single Decision Tree
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(max_depth=10)
tree.fit(X, y)

tree.score(X, y)
rf1.score(X, y)   # usually more stable



# 10. Effect of number of trees
scores = []

for n in [10, 50, 100, 200]:
    rf = RandomForestRegressor(n_estimators=n, random_state=42)
    rf.fit(X_train, y_train)
    scores.append(rf.score(X_test, y_test))

print(scores)



# 11. Create toy regression data for Regression
from sklearn.datasets import make_regression

X, y = make_regression(
    n_samples=100,
    n_features=1,
    noise=10,
    random_state=42
)
```